In [ ]:
# ════════════════════════════════════════════════════
# M3_F04 — PHOTOGRAPHY  |  Cellule 1 : Drive + Install
# ════════════════════════════════════════════════════
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
!pip install -q flask flask-cors
import os, sys
from pathlib import Path
CODEBASE = Path("/content/drive/MyDrive/EXODUS_V2/03_MODE_ASCENSION/F04_PHOTOGRAPHY/CODEBASE")
sys.path.insert(0, str(CODEBASE))
print("Drive monté — CODEBASE :", CODEBASE)

In [ ]:
# ════════════════════════════════════════════════════
# Cellule 2 : Vérification Inputs + Git Pull + Serveur
# ════════════════════════════════════════════════════
import json, subprocess, http.server, threading, os, socket
from pathlib import Path
from google.colab.output import eval_js

DRIVE_ROOT  = Path("/content/drive/MyDrive/EXODUS_V3/M3")
AVATAR_PATH = DRIVE_ROOT / "SHARED" / "avatar.glb"
DECOR_PATH  = DRIVE_ROOT / "SHARED" / "decor.glb"
SPAWN_CFG   = DRIVE_ROOT / "F03_SCENOGRAPHY" / "OUT" / "spawn_config.json"

for label, path in [("avatar.glb", AVATAR_PATH), ("decor.glb", DECOR_PATH), ("spawn_config.json", SPAWN_CFG)]:
    ok = path.exists()
    sz = path.stat().st_size // 1024 if ok else 0
    status = "OK  (" + str(sz) + " KB)" if ok else "ABSENT"
    print(label.ljust(25) + status)

if SPAWN_CFG.exists():
    with open(SPAWN_CFG) as f:
        cfg = json.load(f)
    spawn = cfg.get("spawn")
    scale = cfg.get("scale")
    rot_y = cfg.get("rot_y")
    print("Spawn:", spawn, "| scale:", scale, "| rot_y:", rot_y)

# Git pull
result = subprocess.run(
    ["git", "pull"],
    cwd="/content/drive/MyDrive/EXODUS_V2",
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

# Serveur
CODEBASE_HTTP = "/content/drive/MyDrive/EXODUS_V2/03_MODE_ASCENSION/F04_PHOTOGRAPHY/CODEBASE"
os.chdir(CODEBASE_HTTP)
print("CWD :", os.getcwd())

PORT = 6004

def port_libre(p):
    s = socket.socket()
    try:
        s.bind(("", p))
        s.close()
        return True
    except:
        return False

if port_libre(PORT):
    handler = http.server.SimpleHTTPRequestHandler
    httpd = http.server.HTTPServer(("", PORT), handler)
    thread = threading.Thread(target=httpd.serve_forever, daemon=True)
    thread.start()
    print("Serveur lancé sur port", PORT)
else:
    print("Port", PORT, "déjà actif — OK")

url = eval_js("google.colab.kernel.proxyPort(" + str(PORT) + ")")
print("URL :", url + "/m3_f04_viewer.html")

In [ ]:
# ════════════════════════════════════════════════════
# Cellule 3 : Lancement Flask
# ════════════════════════════════════════════════════
import shutil, threading, time
from google.colab.output import eval_js

LOCAL = Path("/content/m3_f04")
LOCAL.mkdir(exist_ok=True)
for f in CODEBASE.glob("*"):
    shutil.copy(f, LOCAL / f.name)

PORT = 5004
def run_flask():
    os.chdir(str(LOCAL))
    os.system("python m3_f04_flask.py")

t = threading.Thread(target=run_flask, daemon=True)
t.start()
time.sleep(2)
url = eval_js("google.colab.kernel.proxyPort(" + str(PORT) + ")")
print("Viewer F04 :")
print(url)

In [ ]:
# ════════════════════════════════════════════════════
# Cellule 4 : Lire configs générées
# ════════════════════════════════════════════════════
import json
OUT = DRIVE_ROOT / "F04_PHOTOGRAPHY" / "OUT"
for fname in ["camera_config.json", "light_config.json"]:
    p = OUT / fname
    if p.exists():
        with open(p) as f:
            print("=== " + fname + " ===")
            print(json.dumps(json.load(f), indent=2))
    else:
        print(fname + " pas encore généré")